# Owner and Transaction Scale Calculations
Created by Nicholas Polimeni

Updated by Melissa Juarez to include more data cleaning

This models after the fulton entity scale code but removes sales code and focuses only on ownership key creation.

In [2]:
import pandas as pd
import os

pd.set_option('display.max_columns', 150)
pd.options.display.float_format = '{:,.3f}'.format

# set wd one folder back
os.chdir('/Users/melissajuarezc/Documents/GITHUB REPOS/parcel-data-processor/')

In [4]:
# Cleaned digest and sales data from clean_data.ipynb
FILES_PATH = 'output/cobb/'
digest_full = pd.read_csv(
    FILES_PATH + 'DIGEST_cobb_2011_2021_jul022025.csv',
    dtype = {
    "tax_year": "Int64",
    "jur": "Int64",
    "rolltype": "string",
    "taxyr": "Int64",
    "stub": "boolean",
    "parid": "string",
    "n_a": "boolean",
    "lotunit": "string",
    "subblck": "string",
    "user1": "string",
    "acres": "float64",
    "propertyaddr": "string",
    "grossvalue": "Int64",
    "assessvalue": "Int64",
    "hsexemption": "string",
    "distcode": "string",
    "taxpayernum": "string",
    "owner1": "string",
    "owner2": "string",
    "addr1": "string",
    "addr2": "string",
    "addr3": "string",
    "oldowner1": "string",
    "sgexm": "Int64",
    "sbexm": "Int64",
    "cgexm": "Int64",
    "cbexm": "Int64",
    "cfexm": "Int64",
    "ctexm": "Int64",
    "ctbexm": "Int64",
    "stexm": "Int64",
    "sgnet": "Int64",
    "sbnet": "Int64",
    "cgnet": "Int64",
    "cbnet": "Int64",
    "cfnet": "Int64",
    "ctnet": "Int64",
    "ctbnet": "Int64",
    "stnet": "Int64",
    "sgtax": "float64",
    "sbtax": "Int64",
    "cgtax": "float64",
    "cbtax": "float64",
    "cftax": "float64",
    "cttax": "float64",
    "ctbtax": "Int64",
    "sttax": "float64",
    "penalty": "float64",
    "total_tax": "float64",
    "putback": "string",
    "vendlic": "string",
    "btaxyrs": "string",
    "applval": "Int64",
    "applamt": "float64",
    "sgmil": "float64",
    "sbmil": "Int64",
    "cgmil": "float64",
    "cbmil": "float64",
    "cfmil": "float64",
    "ctmil": "float64",
    "ctbmil": "Int64",
    "stmil": "float64",
    "landcode": "string",
    "impcode": "string",
    "landvalue": "Int64",
    "impvalue": "Int64",
    "lendinst": "Int64",
    "statecr": "Int64",
    "countycr": "Int64",
    "schoolcr": "Int64",
    "citycr": "Int64",
    "firecr": "Int64",
    "countyf": "Int64",
    "cityf": "boolean",
    "o_addrtype": "string",
    "o_adrno": "Int64",
    "o_adradd": "string",
    "o_adrdir": "string",
    "o_adrstr": "string",
    "o_adrsuf": "string",
    "o_adrsuf2": "string",
    "o_cityname": "string",
    "o_statecode": "string",
    "o_country": "string",
    "o_postalcode": "string",
    "o_unitdesc": "string",
    "o_unitno": "string",
    "o_addr1": "string",
    "o_addr2": "string",
    "o_addr3": "string",
    "o_zip1": "string",
    "o_zip2": "string",
    "o_carrier_rt": "string",
    "o_postal_indx": "string",
    "l_adrpre": "Int64",
    "l_adrno": "Int64",
    "l_adradd": "string",
    "l_adrdir": "string",
    "l_adrstr": "string",
    "l_adrsuf": "string",
    "l_adrsuf2": "string",
    "l_cityname": "string",
    "l_unitdesc": "string",
    "l_unitno": "string",
    "l_zip1": "Int64",
    "l_zip2": "string",
    "l_loc2": "string",
    "ssexm": "Int64",
    "ssnet": "Int64",
    "sstax": "float64",
    "ssmil": "float64",
    "blank_space": "string",
    "sfexm": "Int64",
    "sfnet": "Int64",
    "sftax": "float64",
    "sfmil": "float64"
}
)

/var/folders/8s/26_f81857z9bqbzzb4yf004w0000gn/T/ipykernel_99687/879180214.py:3: DtypeWarning: Columns (116) have mixed types. Specify dtype option on import or set low_memory=False.
  digest_full = pd.read_csv(


## Modify data to enable aggreggating on entity key

In [5]:
## goal: reduce number of property adrnos that are 0 by accounting for the following data error cases
digest_full['mod_own_adrstr'] = digest_full['o_adrstr'].copy(deep=True)
digest_full['mod_own_adrstr'] = digest_full['mod_own_adrstr'].str.replace(r'[.?]', '', regex=True) ## remove ?s and periods
digest_full['mod_own_adrstr'] = digest_full['mod_own_adrstr'].replace(r'^(?:X+)?$', pd.NA, regex=True) ## remove where cell is "" or only Xs

## for rows where mod_own_adrstr is NA, replace with contents of o_addr1 then o_addr2
digest_full['mod_own_adrstr'] = digest_full['mod_own_adrstr'].fillna(digest_full['o_addr1'])
digest_full['mod_own_adrstr'] = digest_full['mod_own_adrstr'].fillna(digest_full['o_addr2'])

### Identify same owners in parcel data
- Drop any rows without Owner Address
- Create an Owner Address (labeled: "owner_addr") column that is the concatentation of owner address number, owner address string, and owner zip.
- If address string contains numbers, then it is a PO BOX. However, a lot are formatted in different ways, such as P O BOX 123, PO BOX 123, P.O. BOX 123, etc. We can retain the number from the address string, and manually prepend PO BOX, so all will have an identical format.

**Why:** these values get us a highly accurate key for same owner. Owner address string does not contain postfixes like ST, AVE, etc. that might cause issues. Combined with owner number and owner zip, we can say with high confidence that the address is the same while avoiding many common differences amongst the same address (ST vs STREET, etc.). This method is prefered over names which has a higher chance of false positive, and large corporations may operate with differently named subsidaries. This method may also undercount, if a company uses multiple addresses, but this is somewhat unlikely and undercounting is simply an acceptable limitation. It is acceptable since large investors (who would use different addresses) will own so many properties with each subsidary that it will be binned in the correct bin regardless.

In [6]:
# Drop rows w/o owner address; drop those where address is only Xs or ?s
digest_full = digest_full.dropna(subset=["mod_own_adrstr"])

# Re-format PO BOXES
re_box_and_numbers = r".*BOX.*[0-9].*"
re_capture_numbers = r"([0-9]+)"

mask = digest_full["mod_own_adrstr"].str.contains(re_box_and_numbers, regex=True, na=False)

digest_full.loc[mask, "mod_own_adrstr"] = "PO BOX " + digest_full.loc[
    mask, "mod_own_adrstr"
].str.extract(re_capture_numbers)[0]

# clean unit numbers remove #s & extra spaces
digest_full['mod_unitno'] = digest_full['o_unitno'].copy(deep=True)
digest_full['mod_unitno'] = digest_full['mod_unitno'].str.replace(r'[#-]', '', regex=True) ## remove hashtags & hyphens
digest_full['mod_unitno'] = digest_full['mod_unitno'].str.replace(r"\s{1,}", '', regex=True) ## remove spaces
digest_full['mod_unitno'] = digest_full['mod_unitno'].fillna('')  # missing value with ""

In [7]:
mask.sum()

np.int64(109114)

In [8]:
# Print total number of PO BOXES without a number in their address string
re_po_box_no_number = r"^(?!.*\d)[P]+.* BOX.*"
len(digest_full[digest_full["mod_own_adrstr"].str.contains(
    re_po_box_no_number, regex=True, na=False
)][["o_adrno", "mod_own_adrstr"]])

104

In [9]:
# Regex to clean by replacing dots, commas, and multiple spaces
# Also make all strings uppercase (they should be already)

re_dots_commas = r"[.,]+"
re_multiple_spaces = r"\s{2,}"

digest_full["o_addr"] = (
    digest_full["o_adrno"].astype("string").fillna('') + " " +
    digest_full["mod_own_adrstr"] + " " +
    digest_full["mod_unitno"].fillna('') + " " +
    digest_full["o_zip1"].astype("string").fillna('')
).str.replace(
    re_dots_commas,
    "",
    regex=True
).str.replace(
    re_multiple_spaces,
    " ",
    regex=True
).str.strip().str.upper()

In [10]:
digest_full[
    ["o_adrno", "o_adrstr", "mod_own_adrstr", "o_unitno", "mod_unitno", "o_zip1", "o_addr"]
].sample(10)

,o_adrno,o_adrstr,mod_own_adrstr,o_unitno,mod_unitno,o_zip1,o_addr
1401476,3937,SUMMER BREEZE,SUMMER BREEZE,<NA>,,30066,3937 SUMMER BREEZE 30066
1744219,2005,DOBBINS,DOBBINS,<NA>,,30144,2005 DOBBINS 30144
1413168,6242,SHELBURNE PARK,SHELBURNE PARK,<NA>,,30126,6242 SHELBURNE PARK 30126
2397628,60,FOXRIDGE,FOXRIDGE,<NA>,,30067,60 FOXRIDGE 30067
2592373,4709,MYSTIC,MYSTIC,<NA>,,30075,4709 MYSTIC 30075
135110,1618,CORONA,CORONA,<NA>,,30168,1618 CORONA 30168
2425943,4521,KING SPRINGS,KING SPRINGS,<NA>,,30082,4521 KING SPRINGS 30082
2385593,3453,FOX HOLLOW,FOX HOLLOW,<NA>,,30068,3453 FOX HOLLOW 30068
1581167,2433,SPRING LAKE,SPRING LAKE,<NA>,,30062,2433 SPRING LAKE 30062
619257,1681,MOHAWK,MOHAWK,<NA>,,30080,1681 MOHAWK 30080


## Identify corporate owners, create corp owner flags for each record
- grantee, grantor in sales
- own1 in digest

In [11]:
# Any with risk of false positive like "CO" need to have a space prepended or postpended
corp_keywords = [
    'LLC', ' INC', 'LLP', 'L.L.C', 'L.L.P', 'I.N.C', 'L L C',
    'L L P', ' L P', ' LP', 'LTD', ' CORP', 'CORPORATION',
    'COMPANY', ' CO ', 'LIMITED', 'PARTNERSHIP', 'PARTNERSHIPS',
    'ASSOCIATION', 'ASSOC', 'INCORPORATED', 'INCORP',
    'L.T.D', 'LTD', "HOME", "SOLUTIONS"
]

# Make a list of all corp owners -- added own2 as well.
corps = digest_full[
    digest_full["owner1"].apply(lambda x: any([key in str(x) for key in corp_keywords]))
]['owner1'].unique().tolist() + digest_full[
    digest_full["owner2"].apply(lambda x: any([key in str(x) for key in corp_keywords]))
]['owner2'].unique().tolist()

with open("./output/cobb/corp_names.txt", "w") as f:
    f.write("\n".join(corps))

In [ ]:
digest_full["own_corp_flag"] = (
    digest_full["owner1"].isin(corps) | digest_full["owner2"].isin(corps)
).astype(int)

digest_full[['owner1', 'owner2', 'own_corp_flag']].sample(10)

,owner1,owner2,own_corp_flag
1467700,CHAMPAGNE MONIQUE S,<NA>,0
2344148,OSTELL JEANETTE LYNN,<NA>,0
2830984,CRAWFORD LYNETTE M,<NA>,0
2562707,MITCHELL RICHARD M & DEBORAH A,<NA>,0
860415,REED VALERIE & DANIEL,<NA>,0
1130847,LAUREL CREEK COURT HOMEOWNERS,ASSOCIATON INC,1
2123223,PURDUE ORREN L III & RUTH GARRETT,<NA>,0
93246,CAPE CATHERINE A,<NA>,0
749786,AUTO CHLOR SYSTEM OF MID-SOUTH LLC,<NA>,1
1932924,AMOUDOUA FRANCIS A,<NA>,0


## Create a rental property flag

In [13]:
# when owner address is not the same as property address
digest_full["rental_flag"] = 0
digest_full.loc[
    ((digest_full["l_adrno"] != digest_full["o_adrno"])
    & (digest_full["l_adrstr"] != digest_full["o_adrstr"])),
    "rental_flag"
] = 1

In [14]:
digest_full[['l_adrno', 'l_adrstr', 'o_adrno', 'o_adrstr', 'rental_flag']].sample(20)

,l_adrno,l_adrstr,o_adrno,o_adrstr,rental_flag
1981367,1363,FOXHALL,3961,FLOYD,1
2152997,633,INGLEWOOD,6205,RIVER CHASE,1
652194,6310,STONEY,6310,STONEY,0
2526077,1814,TRANQUIL FIELD,1814,TRANQUIL FIELD,0
2721880,1962,SPECTRUM,750,FRANKLIN,1
2684639,4125,FAWN,4125,FAWN,0
1957797,2110,CORSICA,2110,CORSICA,0
2362413,1182,RAMBLEWOOD,1182,RAMBLEWOOD,0
1804281,4874,SYDNEY,4874,SYDNEY,0
2581368,<NA>,VARIOUS,1241,DUBLIN,0


## Create ownership scale table

In [15]:
owned_fulton_yr = pd.DataFrame(
    digest_full.groupby(["taxyr", "o_addr"])["parid"].count()
).rename(columns={"parid": "count_owned_cobb_yr"}).reset_index()
owned_fulton_yr

assoc_owner_names = pd.DataFrame(
    digest_full.groupby(["o_addr"]).agg({"owner1": list})
).rename(columns={"owner1": "assoc_owner_names"}).reset_index()

owner_scale = owned_fulton_yr.merge(
    assoc_owner_names,
    on=["o_addr"],
    how="left"
)

owner_scale.sort_values(by="count_owned_cobb_yr", ascending=False).head(5)

,taxyr,o_addr,count_owned_cobb_yr,assoc_owner_names
1928841,2020,1717 MAIN 2000 75201,1009,"[IH5 PROPERTY BORROWER LP, IH5 PROPERTY BORROW..."
1498325,2018,1717 MAIN 2000 75201,1004,"[IH5 PROPERTY BORROWER LP, IH5 PROPERTY BORROW..."
2147690,2021,1717 MAIN 2000 75201,995,"[IH5 PROPERTY BORROWER LP, IH5 PROPERTY BORROW..."
1711489,2019,1717 MAIN 2000 75201,976,"[IH5 PROPERTY BORROWER LP, IH5 PROPERTY BORROW..."
1286804,2017,1717 MAIN 2000 75201,968,"[IH5 PROPERTY BORROWER LP, IH5 PROPERTY BORROW..."


In [16]:
owner_scale[
    owner_scale["taxyr"] == 2020
].sort_values(by="count_owned_cobb_yr", ascending=False).head(15)

,taxyr,o_addr,count_owned_cobb_yr,assoc_owner_names
1928841,2020,1717 MAIN 2000 75201,1009,"[IH5 PROPERTY BORROWER LP, IH5 PROPERTY BORROW..."
1989476,2020,30601 AGOURA 200 91301,707,"[AH4R1 GA LLC, AH4R GA3, AH4R1 GA LLC, AH4R1 G..."
2101294,2020,8665 HARTFORD 200 85255,526,"[COLFIN AI GA 1 LLC, COLFIN AI GA 1 LLC, COLFI..."
2108965,2020,PO BOX 4090 85261,460,"[FREO GEORGIA LLC, FREO GEORGIA LLC, FREO GEOR..."
1935143,2020,1850 PARKWAY 900 30067,329,"[CERBERUS SFR HOLDINGS LP, CSMA BLT LLC, CSMA ..."
2020949,2020,3820 MANSELL 300 30022,291,"[ASHTON ATLANTA RESIDENTIAL LLC, ASHTON ATLANT..."
2009477,2020,3505 KOGER 400 30096,286,"[RNTR 3 LLC, RNTR 3 LLC, RNTR 3 LLC, RHA 1 LLC..."
1932347,2020,180 STETSON 3650 60601,268,"[HP GEORGIA I LLC, HP GEORGIA I LLC, HP GEORGI..."
2096015,2020,750 CHASTAIN 30066,214,"[KERLEY FAMILY HOMES LLC, KERLEY FAMILY HOMES ..."
1893072,2020,1000 ABERNATHY 260 30328,209,"[BOWERS LORA & ROONEY SHAUN, BEAZER GAIN LLC, ..."


### Identify major institutional investors

In [17]:
import re

owner_keywords = {
    "Amherst": ["AMHERST", "ARVM"],
    "Cerberus": ["CERBERUS", "FKH", "RM1 ", "RMI "],
    "Progress": ["PROGRESS", "FYR"],
    "Invitation": ["INVITATION", "IH "],
    "Colony": ["COLONY", "STARWOOD", "CSH", "CAH "],
    "Sylvan": ["SYLVAN", "RNTR"],
    "Tricon": ["TRICON", "TAH"]
}

for owner in owner_keywords:
    query_str = "|".join(owner_keywords[owner])

    filtered_rows = owner_scale[
        (owner_scale["taxyr"] == 2020) &
        owner_scale["assoc_owner_names"].apply(lambda x: any(
            ((re.search(query_str, name)) for name in x)
        ))
    ]
    
    filtered_rows = filtered_rows[
        filtered_rows["count_owned_cobb_yr"] > 49
    ]
    
    total = filtered_rows["count_owned_cobb_yr"].sum()
    print(f"{owner} owned {total} properties in 2020")
    display(filtered_rows)
    
    addresses = filtered_rows["o_addr"].unique().tolist()
    print(addresses)
    
    names = [set(x) for x in filtered_rows[
        filtered_rows["count_owned_cobb_yr"] > 49
    ]["assoc_owner_names"].to_list()]

    names = set().union(*names)
    names = ", ".join(names)

    with open(f"./output/cobb/ownership_scale/{owner}_names.txt", "w") as f:
        f.write("KEYWORDS: " + query_str + "\n")
        f.write("ADDRESSES: " + ", ".join(addresses) + "\n\n")
        f.write("NAMES\n--------------------\n")
        f.write(names)

TypeError: expected string or bytes-like object, got 'NAType'

## Save owner scale

In [60]:
OUTPUT_PATH = 'output/cobb/'

owner_scale.to_csv(OUTPUT_PATH + 'ownership_scale/owner_scale.csv', index=False)

OSError: Cannot save file into a non-existent directory: 'output/cobb/ownership_scale'

## Save

In [18]:
OUTPUT_PATH = 'output/cobb/'

digest_full.to_csv(OUTPUT_PATH + 'cobb_digest_full_final.csv', index=False)